In [1]:
%load_ext autoreload
import numpy as np
from utils.data_loader import *
from models import *
import torch.optim as optim
import torch
import torch.nn as nn
import l4casadi
import plotly.graph_objects as go
from shred_config_tester import make_shred_config, testing
import plotly.graph_objects as go


In [2]:
c1_picker = torch.tensor(np.loadtxt('data/core_regions/c1_picker.txt'), dtype=torch.float32)
c2_picker = torch.tensor(np.loadtxt('data/core_regions/c2_picker.txt'), dtype=torch.float32)
sus_picker = torch.tensor(np.loadtxt('data/core_regions/sus_picker.txt'), dtype=torch.float32)
outer_wall_picker = torch.tensor(np.loadtxt('data/core_regions/outer_wall_picker.csv'), dtype=torch.float32)
pyro_picker = torch.tensor(np.loadtxt('data/core_regions/pyro_picker.csv'), dtype=torch.float32)

In [3]:
configs = {
    1: [
        ['Power 3D (kW)', 'Input Energy 3D (kWh)', 'Cooling Energy 3D (kWh)', 'Pyro 3D (°C)'], 
        ['zero','init','init','init']
        ],
    2: [['Power 3D (kW)', 'Cooling Power 3D (kW)', 'Net Energy 3D (kWh)', 'Pyro 3D (°C)'],
        ['zero','zero','init','init']
        ],
    3: [['']]
}
shred_models = {}
train_sets = {}
valid_sets = {}
test_sets = {}
managers = {}

Config 1

In [4]:
%autoreload 2
config = 1
mu_keys = configs[config][0]
padding_key = configs[config][1]
shred, train_dataset, valid_dataset, test_dataset, manager = make_shred_config(mu_keys=mu_keys, POD_dim=10, padding_key=padding_key, state_dict_file=f'shred_configs/config{config}.json')
shred_models[config] = shred
train_sets[config] = train_dataset
valid_sets[config] = valid_dataset
test_sets[config] = test_dataset
managers[config] = manager

/home/abhyu/Documents/SINTEF/Gitlab/clean_start/utils/utilities.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.stdev = torch.tensor(np.sqrt(self.var), dtype =torch.float32)


In [140]:
config = 1
shred = shred_models[config]
train_dataset = train_sets[config]
valid_dataset = valid_sets[config]
test_dataset = test_sets[config]
manager = managers[config]
testing(shred, train_dataset, valid_dataset, test_dataset, manager)

-----Training-----
POD coef MS error       :  9.501543045043945
Parseval check          :  PASSED
Temperature MS error    :  2067.911376953125
-----Validation-----
POD coef MS error       :  9.056879997253418
Parseval check          :  PASSED
Temperature MS error    :  1705.6322021484375
-----Testing-----
POD coef MS error       :  9.006897926330566
Parseval check          :  PASSED
Temperature MS error    :  3541.560791015625


In [20]:
from plotly.subplots import make_subplots
fig_list_list = []
for k in range(test_dataset.X.shape[0] // 61):
    show_legend = False if k > 0 else True
    show_legend=True
    pred = shred.post(shred(test_dataset.X[k*61:(k+1)*61]))
    truth = shred.post(test_dataset.Y[k*61:(k+1)*61])
    # C1
    c1_pred = pred@c1_picker.T
    c1_truth = truth@c1_picker.T
    c1_error = c1_pred - c1_truth
    # C2
    c2_pred = pred@c2_picker.T
    c2_truth = truth@c2_picker.T
    c2_error = c2_pred - c2_truth
    # SUS
    sus_pred = pred@sus_picker.T
    sus_truth = truth@sus_picker.T
    sus_error = sus_pred - sus_truth
    
    zone_temps = {'C1': c1_error.detach(), 'C2': c2_error.detach(), 'Sus': sus_error.detach()}#, 'Shell': outer}
    colors = {name:col for name,col in zip(zone_temps.keys(), [['blue', 'rgba(0,100,255,0.2)' ], ['red', 'rgba(255,100,0,0.2)' ], ['green', 'rgba(100,255,0,0.2)' ]])}
    fig_list = []
    for name, zone in zone_temps.items().__reversed__():
        mean = zone.mean(dim=1)
        lower = zone.min(dim=1)[0]
        upper = zone.max(dim=1)[0]
        t = np.arange(len(mean))
        col1, col2 = colors[name]
                
        fig_list += [
            go.Scatter(x=t, y=upper, mode='lines', line=dict(width=0), showlegend=False),
            go.Scatter(x=t, y=lower, mode='lines', line=dict(width=0), fill='tonexty', fillcolor=col2, name='Min–Max range',  showlegend=show_legend),
            go.Scatter(x=t, y=mean, mode='lines', line=dict(color=col1), name=name, showlegend=show_legend),
        ]
    fig = go.Figure(fig_list)
    fig_list_list.append(fig_list)
    fig.update_yaxes(range=(-200,200), title='Error (°C)')
    fig.update_xaxes(title='Timestep')
    fig.update_layout(
    width=800, height=600,
    font=dict(size=20),
    margin=dict(l=70, r=20, t=60, b=60),
    )
    fig.update_layout(title=f'Absolute prediction reconstructoin error, test set {k+1}', legend=dict(x=0.77,y=1,bgcolor='rgba(255,255,255,0.4)'))
    fig.show()   
    fig.write_image(f"plot{k}.pdf", scale=1)  # kaleido export
    fig.write_html(f'test{k}.html')


Config 2

In [8]:
%autoreload 2
config = 2
mu_keys = configs[config][0]
padding_key = configs[config][1]
shred, train_dataset, valid_dataset, test_dataset, manager = make_shred_config(
    mu_keys=mu_keys, 
    POD_dim=10, 
    padding_key=padding_key, state_dict_file=f'shred_configs/config{config}.json')
shred_models[config] = shred
train_sets[config] = train_dataset
valid_sets[config] = valid_dataset
test_sets[config] = test_dataset
managers[config] = manager

/home/abhyu/Documents/SINTEF/Gitlab/clean_start/utils/utilities.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.stdev = torch.tensor(np.sqrt(self.var), dtype =torch.float32)


In [9]:
config = 2
shred = shred_models[config]
train_dataset = train_sets[config]
valid_dataset = valid_sets[config]
test_dataset = test_sets[config]
manager = managers[config]
testing(shred, train_dataset, valid_dataset, test_dataset, manager)

-----Training-----
POD coef MS error       :  0.03778630867600441
Parseval check          :  PASSED
Relative MS recon error :  0.013233139179646969
-----Validation-----
POD coef MS error       :  0.03237107768654823
Parseval check          :  PASSED
Relative MS recon error :  0.011253907345235348
-----Testing-----
POD coef MS error       :  0.025436220690608025
Parseval check          :  PASSED
Relative MS recon error :  0.008742080070078373


In [112]:
train_dataset.Y.min()

tensor(0.)

In [18]:
from plotly.subplots import make_subplots
fig_list_list = []
for k in range(test_dataset.X.shape[0] // 61):
    show_legend = False if k > 0 else True
    show_legend=True
    pred = shred.post(shred(test_dataset.X[k*61:(k+1)*61]))
    truth = shred.post(test_dataset.Y[k*61:(k+1)*61])
    # C1
    c1_pred = pred@c1_picker.T
    c1_truth = truth@c1_picker.T
    c1_error = c1_pred - c1_truth
    # C2
    c2_pred = pred@c2_picker.T
    c2_truth = truth@c2_picker.T
    c2_error = c2_pred - c2_truth
    # SUS
    sus_pred = pred@sus_picker.T
    sus_truth = truth@sus_picker.T
    sus_error = sus_pred - sus_truth
    
    zone_temps = {'C1': c1_error.detach(), 'C2': c2_error.detach(), 'Sus': sus_error.detach()}#, 'Shell': outer}
    colors = {name:col for name,col in zip(zone_temps.keys(), [['blue', 'rgba(0,100,255,0.2)' ], ['red', 'rgba(255,100,0,0.2)' ], ['green', 'rgba(100,255,0,0.2)' ]])}
    fig_list = []
    for name, zone in zone_temps.items().__reversed__():
        mean = zone.mean(dim=1)
        lower = zone.min(dim=1)[0]
        upper = zone.max(dim=1)[0]
        t = np.arange(len(mean))
        col1, col2 = colors[name]
                
        fig_list += [
            go.Scatter(x=t, y=upper, mode='lines', line=dict(width=0), showlegend=False),
            go.Scatter(x=t, y=lower, mode='lines', line=dict(width=0), fill='tonexty', fillcolor=col2, name='Min–Max range',  showlegend=show_legend),
            go.Scatter(x=t, y=mean, mode='lines', line=dict(color=col1), name=name, showlegend=show_legend),
        ]
    fig = go.Figure(fig_list)
    fig_list_list.append(fig_list)
    fig.update_yaxes(range=(-200,200), title='Error (°C)')
    fig.update_xaxes(title='Timestep')
    fig.update_layout(
    width=800, height=600,
    font=dict(size=20),
    margin=dict(l=70, r=20, t=60, b=60),
    )
    fig.update_layout(title=f'Absolute prediction reconstructoin error, test set {k+1}', legend=dict(x=0.77,y=1,bgcolor='rgba(255,255,255,0.4)'))
    fig.show()   
    fig.write_image(f"plot{k}.pdf", scale=1)  # kaleido export
    fig.write_html(f'test{k}.html')


In [87]:
def minmaxmeanfigs(c1_error, c2_error, sus_error, show_legend=False, legend_group='2'):
    zone_temps = {'C1': c1_error.detach(), 'C2': c2_error.detach(), 'Sus': sus_error.detach()}#, 'Shell': outer}
    colors = {name:col for name,col in zip(zone_temps.keys(), [['blue', 'rgba(0,100,255,0.2)' ], ['red', 'rgba(255,100,0,0.2)' ], ['green', 'rgba(100,255,0,0.2)' ]])}
    fig_list = []
    for name, zone in zone_temps.items().__reversed__():
        mean = zone.mean(dim=1)
        lower = zone.min(dim=1)[0]
        upper = zone.max(dim=1)[0]
        t = np.arange(len(mean))
        col1, col2 = colors[name]
                
        fig_list += [
            go.Scatter(x=t, y=upper, mode='lines', line=dict(width=0), showlegend=False),
            go.Scatter(x=t, y=lower, mode='lines', line=dict(width=0), fill='tonexty', fillcolor=col2, name='Min–Max range',  showlegend=show_legend, legendgroup=legend_group),
            go.Scatter(x=t, y=mean, mode='lines', line=dict(color=col1), name=name, showlegend=show_legend, legendgroup=legend_group),
        ]
    return fig_list

In [107]:
for k in range(3):
    pred1 = shred_models[1].post(shred_models[1](test_sets[1].X[k*61:(k+1)*61]))
    truth1 = shred_models[1].post(test_sets[1].Y[k*61:(k+1)*61])

    pred2 = shred_models[2].post(shred_models[2](test_sets[2].X[k*61:(k+1)*61]))
    truth2 = shred_models[2].post(test_sets[2].Y[k*61:(k+1)*61])
    # C1
    c1_truth = truth1 @ c1_picker.T
    c1_error1 = (pred1 - truth1) @ c1_picker.T
    c1_error2 = (pred2 - truth2) @ c1_picker.T
    # C2
    c2_truth = truth1 @ c2_picker.T
    c2_error1 = (pred1 - truth1) @ c2_picker.T
    c2_error2 = (pred2 - truth2) @ c2_picker.T
    # SUS
    sus_truth = truth1 @ sus_picker.T
    sus_error1 = (pred1 - truth1) @ sus_picker.T
    sus_error2 = (pred2 - truth2) @ sus_picker.T

    rows,cols=2,2
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=['Inputs (sensors)', 'Temperature (ground truth)', 'Config 1', 'Config 2'])

    fig_list1 = minmaxmeanfigs(c1_error1, c2_error1, sus_error1, show_legend=True, legend_group='1')
    fig_list2 = minmaxmeanfigs(c1_error2, c2_error2, sus_error2, legend_group='1')
    fig_listY = minmaxmeanfigs(c1_truth, c2_truth, sus_truth, legend_group='1' )


    input_names = [('Heating power',True,'#1f77b4'), ('Heating energy',True,'#ff7f0e'), ('Cooling energy',True,'#2ca02c'), ('Pyro',True,'#d62728')]
    for i in range(test_sets[1].X.shape[-1]):
        j = test_sets[2].X.shape[-1] - 1 - i
        fig.add_trace(
            go.Scatter(x=np.arange(len(c1_error)), y=test_sets[1].X[k*61:(k+1)*61,-1,j], name=input_names[j][0], showlegend=input_names[j][1], line=dict(color=(input_names[j][-1])), legendgroup='2'),
            row=1, col=1)
        
    input_names = [('Heating power',False,'#1f77b4'), ('Cooling power',True,'#9467bd'), ('Net energy',True,'#8c564b'), ('Pyro',False,'#d62728')]
    for i in range(test_sets[2].X.shape[-1]):
        j = test_sets[2].X.shape[-1] - 1 - i
        fig.add_trace(
            go.Scatter(x=np.arange(len(c1_error)), y=test_sets[2].X[k*61:(k+1)*61,-1,j], name=input_names[j][0], showlegend=input_names[j][1], line=dict(color=(input_names[j][-1])), legendgroup='2'),
            row=1, col=1)
        

    for trace in fig_listY:
        fig.add_trace(trace, row=1,col=2)
    for trace in fig_list1:
        fig.add_trace(trace, row=2,col=1)
    for trace in fig_list2:
        fig.add_trace(trace, row=2,col=2)

    fig.update_yaxes(range=(-0.05,1.05), title='Normalised input', row=1, col=1)
    fig.update_yaxes(title='Temperature (°C)', row=1, col=2)
    fig.update_yaxes(range=(-200,200), title='Error (°C)', row=2, col=1)
    fig.update_yaxes(range=(-200,200), title='Error (°C)', row=2, col=2)
    fig.update_xaxes(title='Timestep')
    fig.update_layout(
    width=800, height=600,
    font=dict(size=11),  
    margin=dict(l=70, r=20, t=60, b=60),
    title=f'Test dataset {k+1} summary',
    legend={'traceorder':'reversed'},
    )
    fig.show()

In [36]:
test_sets[1].X[:61,-1,:].shape
np.array(len(c1_error))

array(61)